## UTD-MHAD Data Preprocessing notebook
### Step 0. Setup the paths and env variables

In [1]:
# =========================
# STEP 0 — Setup & contracts
# =========================
from pathlib import Path
import json, sys, re
import numpy as np
import pandas as pd
import scipy.io as sio
from tqdm import tqdm

ROOT = Path("/home/aidan/IMU_LM_Data")
sys.path.insert(0, str(ROOT))

from UTILS.helpers import (
    convert_unit,           # g -> m/s^2, deg/s -> rad/s
    _canon,
)

BASE    = ROOT / "data"
RAW     = BASE / "raw_data" / "UTD-MHAD" / "Inertial"
CLEANED = BASE / "cleaned_premerge"
MERGED  = BASE / "merged_dataset"

SCHEMA_PATH       = ROOT / "Unification" / "schemas" / "continuous_stream_schema.json"
ACTIVITY_MAP_PATH = ROOT / "Unification" / "schemas" / "activity_mapping.json"

SCHEMA       = json.loads(SCHEMA_PATH.read_text())
ACT_MAP_FULL = json.loads(ACTIVITY_MAP_PATH.read_text())

UNKNOWN_ID = int(ACT_MAP_FULL.get("unknown_activity_id", 9000))
TARGET_HZ  = int(SCHEMA.get("rate_hz", 50))

RAW2ID  = { _canon(k): int(v) for k, v in ACT_MAP_FULL.get("mapping", {}).items() }
ID2NAME = { int(x["id"]): x["name"] for x in ACT_MAP_FULL["label_set"] }

print("RAW dir   :", RAW)
print("CLEANED   :", CLEANED)
print("TARGET_HZ :", TARGET_HZ)
print("UNKNOWN_ID:", UNKNOWN_ID)

RAW dir   : /home/aidan/IMU_LM_Data/data/raw_data/UTD-MHAD/Inertial
CLEANED   : /home/aidan/IMU_LM_Data/data/cleaned_premerge
TARGET_HZ : 50
UNKNOWN_ID: 9000


### Step 1. Ingest, preprocess and map the data

In [2]:
# ============================
# STEP 1 — Load UTD-MHAD inertial .mat files (wrist only, actions 1-21)
#   - .mat variable: 'd_iner' -> (N, 6) array
#     cols 0-2: accelerometer in g  -> convert to m/s^2
#     cols 3-5: gyroscope in deg/s  -> convert to rad/s
#   - Already at 50 Hz natively — no resampling needed
#   - File naming: a{action}_s{subject}_t{trial}_inertial.mat
#   - Only actions 1-21 (right wrist); actions 22-27 (right thigh) are dropped
# ============================

MAX_WRIST_ACTION = 21  # actions 1-21 are wrist-worn

ACTION_LABELS = {
    1:  "swipe_left",
    2:  "swipe_right",
    3:  "wave",
    4:  "clap",
    5:  "throw",
    6:  "arm_cross",
    7:  "basketball_shoot",
    8:  "draw_x",
    9:  "draw_circle_cw",
    10: "draw_circle_ccw",
    11: "draw_triangle",
    12: "bowling",
    13: "boxing",
    14: "baseball_swing",
    15: "tennis_swing",
    16: "arm_curl",
    17: "tennis_serve",
    18: "push",
    19: "knock",
    20: "catch",
    21: "pickup_and_throw",
}

G_M_S2     = 9.80665
DEG_TO_RAD = np.pi / 180.0
STEP_NS    = int(1e9 // TARGET_HZ)  # 20_000_000 ns at 50 Hz

def parse_filename(fname: str):
    """Parse a{action}_s{subject}_t{trial}_inertial.mat -> (action, subject, trial) ints."""
    m = re.match(r"a(\d+)_s(\d+)_t(\d+)_inertial\.mat$", fname)
    if m:
        return int(m.group(1)), int(m.group(2)), int(m.group(3))
    return None, None, None

def load_utdmhad_wrist(raw_dir: Path) -> pd.DataFrame:
    files = sorted(raw_dir.glob("a*_s*_t*_inertial.mat"))
    print(f"[UTD-MHAD] Found {len(files)} .mat files under {raw_dir}")

    sessions = []
    skipped_thigh = 0
    skipped_parse = 0
    skipped_empty = 0

    for f in tqdm(files, desc="UTD-MHAD files"):
        action, subject, trial = parse_filename(f.name)
        if action is None:
            skipped_parse += 1
            continue

        # Filter: keep only wrist actions (1-21)
        if action > MAX_WRIST_ACTION:
            skipped_thigh += 1
            continue

        mat = sio.loadmat(str(f))
        if "d_iner" not in mat:
            print(f"[WARN] {f.name} has no 'd_iner' key — skipping")
            skipped_empty += 1
            continue

        data = mat["d_iner"]  # (N, 6): acc_x/y/z in g, gyro_x/y/z in deg/s
        if data.ndim != 2 or data.shape[1] < 6:
            print(f"[WARN] {f.name} unexpected shape {data.shape} — skipping")
            skipped_empty += 1
            continue

        n = data.shape[0]

        # Unit conversion
        acc_x = (data[:, 0] * G_M_S2).astype(np.float32)
        acc_y = (data[:, 1] * G_M_S2).astype(np.float32)
        acc_z = (data[:, 2] * G_M_S2).astype(np.float32)
        gyro_x = (data[:, 3] * DEG_TO_RAD).astype(np.float32)
        gyro_y = (data[:, 4] * DEG_TO_RAD).astype(np.float32)
        gyro_z = (data[:, 5] * DEG_TO_RAD).astype(np.float32)

        # Synthetic 50 Hz timestamps (session-relative)
        timestamp_ns = (np.arange(n, dtype=np.int64) * STEP_NS)

        subject_id = f"S{subject:02d}"
        session_id = f"a{action:02d}_s{subject:02d}_t{trial:02d}"
        label = ACTION_LABELS[action]

        out = pd.DataFrame({
            "dataset":                "utd_mhad",
            "subject_id":             subject_id,
            "session_id":             session_id,
            "timestamp_ns":           timestamp_ns,
            "acc_x":                  acc_x,
            "acc_y":                  acc_y,
            "acc_z":                  acc_z,
            "gyro_x":                gyro_x,
            "gyro_y":                gyro_y,
            "gyro_z":                gyro_z,
            "dataset_activity_id":    np.int16(action),
            "dataset_activity_label": label,
        })
        sessions.append(out)

    print(f"\n[UTD-MHAD] Loaded: {len(sessions)} wrist trials")
    print(f"[UTD-MHAD] Skipped thigh (a>21): {skipped_thigh}")
    print(f"[UTD-MHAD] Skipped parse errors:  {skipped_parse}")
    print(f"[UTD-MHAD] Skipped empty/bad:     {skipped_empty}")

    if not sessions:
        return pd.DataFrame()

    raw = pd.concat(sessions, ignore_index=True)

    print(f"\n=== RAW SUMMARY (UTD-MHAD wrist) ===")
    print(f"Rows: {len(raw):,}")
    print(f"Subjects: {raw['subject_id'].nunique()}")
    print(f"Sessions: {raw['session_id'].nunique()}")
    print(f"Actions:  {raw['dataset_activity_label'].nunique()}")
    print(f"\nTop native labels:")
    print(raw["dataset_activity_label"].value_counts())

    return raw

utdmhad_native = load_utdmhad_wrist(RAW)
utdmhad_native.head(3)

[UTD-MHAD] Found 861 .mat files under /home/aidan/IMU_LM_Data/data/raw_data/UTD-MHAD/Inertial


UTD-MHAD files: 100%|██████████| 861/861 [00:00<00:00, 3099.32it/s]



[UTD-MHAD] Loaded: 671 wrist trials
[UTD-MHAD] Skipped thigh (a>21): 190
[UTD-MHAD] Skipped parse errors:  0
[UTD-MHAD] Skipped empty/bad:     0

=== RAW SUMMARY (UTD-MHAD wrist) ===
Rows: 119,897
Subjects: 8
Sessions: 671
Actions:  21

Top native labels:
dataset_activity_label
pickup_and_throw    7473
bowling             6801
baseball_swing      6453
draw_triangle       6432
draw_circle_cw      6327
draw_circle_ccw     6208
tennis_serve        5897
knock               5801
boxing              5719
tennis_swing        5632
wave                5587
arm_cross           5496
draw_x              5387
clap                5215
push                5206
swipe_left          5150
basketball_shoot    5147
catch               4999
arm_curl            4993
throw               4990
swipe_right         4984
Name: count, dtype: int64


,dataset,subject_id,session_id,timestamp_ns,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,dataset_activity_id,dataset_activity_label
0,utd_mhad,S01,a10_s01_t01,0,-9.332597,-4.082116,-0.893033,-0.317623,-0.151351,0.150285,10,draw_circle_ccw
1,utd_mhad,S01,a10_s01_t01,20000000,-9.301470,-4.173092,-1.163579,-0.326683,-0.136962,0.112447,10,draw_circle_ccw
2,utd_mhad,S01,a10_s01_t01,40000000,-9.222467,-4.177878,-1.381453,-0.373580,-0.099124,0.038904,10,draw_circle_ccw


### Step 2. Map the data and audit the mapping

In [3]:
# ============================================
# STEP 2 — Mapping audit (native → global)
# ============================================
if utdmhad_native.empty:
    raise SystemExit("No UTD-MHAD rows after STEP 1. Check RAW path / .mat files.")

raw_counts = (
    utdmhad_native["dataset_activity_label"]
    .astype("string")
    .map(_canon)
    .value_counts()
    .rename_axis("raw_label")
    .reset_index(name="count")
)

raw_counts["mapped_id"] = raw_counts["raw_label"].map(RAW2ID).fillna(UNKNOWN_ID).astype(int)
raw_counts["mapped_nm"] = raw_counts["mapped_id"].map(lambda x: ID2NAME.get(int(x), "other"))

unmapped = raw_counts.loc[raw_counts["mapped_id"] == UNKNOWN_ID]
print(f"[UTD-MHAD] Unique raw labels: {len(raw_counts)} | Unmapped: {len(unmapped)}")

total_ct = int(raw_counts["count"].sum())
mapped_ct = int(raw_counts.loc[raw_counts["mapped_id"] != UNKNOWN_ID, "count"].sum())
print(f"coverage={100.0*mapped_ct/max(total_ct,1):.2f}%  (mapped={mapped_ct:,}/{total_ct:,})")

if not unmapped.empty:
    print("\nUnmapped raw labels (add to activity_mapping.json):")
    print(unmapped.sort_values("count", ascending=False).to_string(index=False))

raw_counts

[UTD-MHAD] Unique raw labels: 21 | Unmapped: 0
coverage=100.00%  (mapped=119,897/119,897)


,raw_label,count,mapped_id,mapped_nm
0,pickup_and_throw,7473,21,sports_play
1,bowling,6801,21,sports_play
2,baseball_swing,6453,21,sports_play
3,draw_triangle,6432,18,adl_misc_gesture
4,draw_circle_cw,6327,18,adl_misc_gesture
5,draw_circle_ccw,6208,18,adl_misc_gesture
6,tennis_serve,5897,21,sports_play
7,knock,5801,18,adl_misc_gesture
8,boxing,5719,21,sports_play
9,tennis_swing,5632,21,sports_play


### Step 3. Build and clean dataset in stream json format

In [4]:
# =========================================================
# STEP 3 — Build schema-ordered continuous_stream (v3) df
# =========================================================
def to_continuous_stream_utdmhad(df_native: pd.DataFrame) -> pd.DataFrame:
    if df_native.empty:
        return pd.DataFrame(columns=[c["name"] for c in SCHEMA["columns"]])

    raw_key = df_native["dataset_activity_label"].astype("string").map(_canon)
    gid     = raw_key.map(RAW2ID).fillna(UNKNOWN_ID).astype("int16")
    glabel  = gid.map(lambda x: ID2NAME.get(int(x), "other")).astype("string")

    out = pd.DataFrame({
        "dataset":     df_native["dataset"].astype("string"),
        "subject_id":  df_native["subject_id"].astype("string"),
        "session_id":  df_native["session_id"].astype("string"),
        "timestamp_ns": df_native["timestamp_ns"].astype("int64"),

        "acc_x": df_native["acc_x"].astype("float32"),
        "acc_y": df_native["acc_y"].astype("float32"),
        "acc_z": df_native["acc_z"].astype("float32"),
        "gyro_x": df_native["gyro_x"].astype("float32"),
        "gyro_y": df_native["gyro_y"].astype("float32"),
        "gyro_z": df_native["gyro_z"].astype("float32"),

        "global_activity_id": gid,
        "global_activity_label": glabel,

        "dataset_activity_id": df_native["dataset_activity_id"].astype("int16"),
        "dataset_activity_label": df_native["dataset_activity_label"].astype("string"),
    })

    order = [c["name"] for c in SCHEMA["columns"]]
    return out[order]

utdmhad_df = to_continuous_stream_utdmhad(utdmhad_native)
print("UTD-MHAD unified rows:", len(utdmhad_df))
utdmhad_df.head(3)

UTD-MHAD unified rows: 119897


,dataset,subject_id,session_id,timestamp_ns,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,global_activity_id,global_activity_label,dataset_activity_id,dataset_activity_label
0,utd_mhad,S01,a10_s01_t01,0,-9.332597,-4.082116,-0.893033,-0.317623,-0.151351,0.150285,18,adl_misc_gesture,10,draw_circle_ccw
1,utd_mhad,S01,a10_s01_t01,20000000,-9.301470,-4.173092,-1.163579,-0.326683,-0.136962,0.112447,18,adl_misc_gesture,10,draw_circle_ccw
2,utd_mhad,S01,a10_s01_t01,40000000,-9.222467,-4.177878,-1.381453,-0.373580,-0.099124,0.038904,18,adl_misc_gesture,10,draw_circle_ccw


### Step 4. Audit check the unified frame

In [5]:
# ==========================================
# STEP 4 — Contract checks & quick QA
# ==========================================
groups = ["subject_id", "session_id"]

print("Subjects:", utdmhad_df["subject_id"].nunique(),
      "| Sessions:", utdmhad_df["session_id"].nunique())

# Monotonic timestamp per (subject, session)
viol = 0
for (_sid, _sess), g in utdmhad_df.groupby(groups, sort=False):
    ts = g["timestamp_ns"].to_numpy()
    if ts.size and not np.all(np.diff(ts) >= 0):
        viol += 1
print("Monotonic violations (groups):", viol)

# Approx Hz from ns timestamps
def est_hz_ns(ts_ns: pd.Series):
    arr = ts_ns.to_numpy()
    if arr.size < 3:
        return np.nan
    dt = np.diff(arr) / 1e9
    dt = dt[(dt > 0) & np.isfinite(dt)]
    return float(np.median(1.0 / dt)) if dt.size else np.nan

hz = utdmhad_df.groupby(groups)["timestamp_ns"].apply(est_hz_ns)
print(f"Median Hz: {np.nanmedian(hz.values):.2f} (target={SCHEMA['rate_hz']})")

# Required-not-null coverage
req = SCHEMA["expectations"]["required_not_null"]
pct = utdmhad_df[req].notnull().all(axis=1).mean() * 100
print(f"Rows meeting required-not-null: {pct:.2f}%")

# Global mapping coverage
cov = (utdmhad_df["global_activity_id"] != UNKNOWN_ID).mean() * 100
print(f"Global mapping coverage: {cov:.1f}% (unknown={UNKNOWN_ID})")

print("\nTop-15 dataset_activity_label:")
print(utdmhad_df["dataset_activity_label"].value_counts().head(15))

print("\nTop-15 global labels:")
print(utdmhad_df["global_activity_label"].value_counts().head(15))

Subjects: 8 | Sessions: 671
Monotonic violations (groups): 0
Median Hz: 50.00 (target=50)
Rows meeting required-not-null: 100.00%
Global mapping coverage: 100.0% (unknown=9000)

Top-15 dataset_activity_label:
dataset_activity_label
pickup_and_throw    7473
bowling             6801
baseball_swing      6453
draw_triangle       6432
draw_circle_cw      6327
draw_circle_ccw     6208
tennis_serve        5897
knock               5801
boxing              5719
tennis_swing        5632
wave                5587
arm_cross           5496
draw_x              5387
clap                5215
push                5206
Name: count, dtype: Int64

Top-15 global labels:
global_activity_label
adl_misc_gesture    56587
sports_play         53111
exercise_upper      10199
Name: count, dtype: Int64


### Step 5. Save outputs

In [6]:
# ============================
# STEP 5 — Save
# ============================
out_path = CLEANED / "utd_mhad_clean_data.parquet"
out_path.parent.mkdir(parents=True, exist_ok=True)
utdmhad_df.to_parquet(out_path, index=False)
print("Saved →", out_path)

Saved → /home/aidan/IMU_LM_Data/data/cleaned_premerge/utd_mhad_clean_data.parquet


In [ ]:
# ============================
# SCRATCH — Quick plot: pick a subject & see 10s of each action trial
# ============================
import matplotlib.pyplot as plt

# ---- CONFIG: change these ----
PLOT_SUBJECT = "S01"
MAX_SECONDS  = 10.0
# ------------------------------

MAX_NS = int(MAX_SECONDS * 1e9)

sub = utdmhad_df[utdmhad_df["subject_id"] == PLOT_SUBJECT].copy()
sessions = sub["session_id"].unique()

print(f"Subject {PLOT_SUBJECT}: {len(sessions)} sessions")

# pick up to 6 sessions to tile
show = sessions[:6]

fig, axes = plt.subplots(len(show), 2, figsize=(16, 3 * len(show)), sharex=False)
if len(show) == 1:
    axes = axes[np.newaxis, :]

for i, sess in enumerate(show):
    chunk = sub[sub["session_id"] == sess]
    chunk = chunk[chunk["timestamp_ns"] <= MAX_NS]
    t_s = chunk["timestamp_ns"].to_numpy() / 1e9
    lbl = chunk["dataset_activity_label"].iloc[0]

    # Accel
    ax = axes[i, 0]
    ax.plot(t_s, chunk["acc_x"], label="acc_x", alpha=0.8)
    ax.plot(t_s, chunk["acc_y"], label="acc_y", alpha=0.8)
    ax.plot(t_s, chunk["acc_z"], label="acc_z", alpha=0.8)
    ax.set_ylabel("m/s²")
    ax.set_title(f"{sess} — {lbl} (accel)")
    ax.legend(loc="upper right", fontsize=7)
    ax.grid(True, alpha=0.3)

    # Gyro
    ax = axes[i, 1]
    ax.plot(t_s, chunk["gyro_x"], label="gyro_x", alpha=0.8)
    ax.plot(t_s, chunk["gyro_y"], label="gyro_y", alpha=0.8)
    ax.plot(t_s, chunk["gyro_z"], label="gyro_z", alpha=0.8)
    ax.set_ylabel("rad/s")
    ax.set_title(f"{sess} — {lbl} (gyro)")
    ax.legend(loc="upper right", fontsize=7)
    ax.grid(True, alpha=0.3)

axes[-1, 0].set_xlabel("Time (s)")
axes[-1, 1].set_xlabel("Time (s)")
fig.suptitle(f"UTD-MHAD — Subject {PLOT_SUBJECT} (first {MAX_SECONDS}s)", fontsize=14)
fig.tight_layout()
plt.show()